In [3]:
# !pip install deepfilternet==0.5.6

In [4]:
import os
from dotenv import load_dotenv
_ = load_dotenv()


from huggingface_hub import login
login(token=os.environ['HUGGINGFACEHUB_API_TOKEN'])

In [87]:
# Download DF model

from huggingface_hub import snapshot_download

# Specify the model repository ID
repo_id = "hshr/DeepFilterNet2"

# Download the model snapshot
snapshot_download(
    repo_id=repo_id, 
    repo_type="space",
    allow_patterns=["DeepFilterNet2/*"],
    local_dir="/home/nampv1/projects/asr/asr_ft/enhance_audio/DeepFilterNet2_model"
)


Fetching 2 files: 100%|██████████| 2/2 [00:05<00:00,  2.57s/it]


'/home/nampv1/projects/asr/asr_ft/enhance_audio/DeepFilterNet2_model'

In [5]:
def listen_audio(waveform, sr=16000):
    """
    Nghe audio từ waveform tensor hoặc numpy array trong notebook.
    
    Args:
        waveform: torch.Tensor 1D/2D hoặc np.ndarray
        sr: int, sampling rate
    """
    import torch
    from IPython.display import Audio, display
    
    
    # Nếu tensor, chuyển sang numpy
    if isinstance(waveform, torch.Tensor):
        waveform = waveform.detach().cpu().numpy()
    
    # Nếu 2D (stereo), transpose về (num_samples, num_channels)
    if waveform.ndim == 2:
        waveform = waveform.T  # [num_samples, num_channels]
    
    display(Audio(waveform, rate=sr))



In [23]:
def listen_audio(waveform, sr=16000):
    """
    Play audio from a waveform tensor or numpy array in a notebook.
    waveform: torch.Tensor or np.ndarray
              shape: [channels, time] or [time]
    sr: sample rate
    """
    import torch
    import numpy as np
    from IPython.display import Audio, display

    # Convert torch tensor to numpy
    if isinstance(waveform, torch.Tensor):
        waveform = waveform.detach().cpu().numpy()

    # Ensure float32
    waveform = waveform.astype(np.float32)

    # Clip to [-1, 1] to avoid WAV writing errors
    waveform = np.clip(waveform, -1.0, 1.0)

    # If stereo (2D), transpose to [num_samples, num_channels]
    if waveform.ndim == 2:
        waveform = waveform.T

    display(Audio(waveform, rate=sr))


In [41]:
import numpy as np
import soundfile as sf
import torchaudio

# Load clean audio
clean, sr = torchaudio.load("/home/nampv1/projects/asr/asr_ft/data/examples/audio.wav", )

print("sr:", sr)

# Generate Gaussian noise
# Option 1: Use torch.randn (standard normal, mean=0, std=1)
noise = 0.02 * torch.randn_like(clean)


# Mix noise with clean signal
noisy = clean + noise

# # Save noisy audio
torchaudio.save("clean.wav", clean, sr)


sr: 16000


/home/nampv1/anaconda3/envs/asr/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


In [ ]:
save_audio("clean.wav", enhanced_audio, sr)

In [35]:
def listen_audio(
    waveform,
    sr=16000,
    normalize=False,
    auto_clipping=True,
    try_save_to_file=True,
):
    """
    Robust notebook audio player for torch.Tensor or np.ndarray.
    - Accepts shapes:
        * 1D: [num_samples]
        * 2D: [channels, time] or [time, channels]
    - Ensures dtype/shape compatible with IPython.display.Audio.
    - If direct playback fails, falls back to writing a temporary WAV (uses soundfile, torchaudio, or scipy if available).
    Args:
        waveform: torch.Tensor or np.ndarray
        sr: sample rate (int)
        normalize: if True, scale audio so max absolute == 1.0
        auto_clipping: if True, clip values to [-1, 1] before playback
        try_save_to_file: if True, on failure try to write a temporary file and play that
    """
    import numpy as np
    from IPython.display import Audio, display
    import tempfile, os, traceback

    # lazy import torch to avoid hard dependency
    try:
        import torch
        is_torch_tensor = isinstance(waveform, torch.Tensor)
    except Exception:
        torch = None
        is_torch_tensor = False

    # Convert torch -> numpy
    if is_torch_tensor:
        waveform = waveform.detach().cpu().numpy()

    if not isinstance(waveform, np.ndarray):
        waveform = np.array(waveform)

    # Normalize dimensions:
    # Accept [channels, time] -> convert to [time, channels]
    # Accept [time, channels] -> use as-is
    # Accept 1D -> use as-is
    if waveform.ndim == 2:
        # determine whether shape is (channels, time) or (time, channels)
        # heuristic: if first dim is small (1 or 2) treat as (channels, time)
        if waveform.shape[0] <= 2 and waveform.shape[0] < waveform.shape[1]:
            waveform = waveform.T  # -> [time, channels]
        else:
            # could already be [time, channels]; leave as-is
            pass
    elif waveform.ndim > 2:
        raise ValueError(f"Unsupported waveform ndim={waveform.ndim}. Expect 1D or 2D array.")

    # Ensure sr is int and sane
    try:
        sr = int(sr)
    except Exception:
        raise ValueError("sr must be an integer sample rate.")

    # Convert dtype to float32 for Audio if numeric floats
    if np.issubdtype(waveform.dtype, np.floating):
        waveform = waveform.astype(np.float32)
    else:
        # if integer PCM (e.g., int16), convert to float32 in [-1,1]
        info = np.iinfo(waveform.dtype)
        waveform = waveform.astype(np.float32) / float(info.max if info.max > 0 else 1)

    # Optional normalize
    if normalize:
        peak = float(np.abs(waveform).max()) if waveform.size > 0 else 0.0
        if peak > 0:
            waveform = waveform / peak

    # Clip if requested
    if auto_clipping:
        waveform = np.clip(waveform, -1.0, 1.0)

    # Try direct playback first
    try:
        display(Audio(waveform, rate=sr))
        return
    except Exception as e:
        # capture error and continue to fallback
        err = e
        tb = traceback.format_exc()

    # Fallback: write temporary WAV using available writer libs
    if not try_save_to_file:
        # re-raise the original error with some debug context
        raise RuntimeError(
            "Direct playback failed and fallback writing disabled.\n"
            f"Original error: {err}\n\nTraceback:\n{tb}"
        )

    # Try soundfile (pysoundfile)
    tmpfile = None
    last_exc = None
    try:
        import soundfile as sf

        tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        tmpfile = tmp.name
        tmp.close()
        # soundfile expects shape (frames, channels) or (frames,), dtype float32 in [-1,1]
        sf.write(tmpfile, waveform, sr, subtype="PCM_16")
        display(Audio(tmpfile, rate=sr))
        return
    except Exception as e_sf:
        last_exc = e_sf

    # Try torchaudio.save if waveform came from torch or torchaudio present
    try:
        import torchaudio
        import torch as _torch  # may re-import
        # torchaudio.save expects tensor shape [channels, time]
        if waveform.ndim == 1:
            wav_to_save = _torch.from_numpy(waveform).unsqueeze(0)
        else:
            # waveform currently [time, channels] -> transpose to [channels, time]
            wav_to_save = _torch.from_numpy(waveform.T)
        tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        tmpfile = tmp.name
        tmp.close()
        torchaudio.save(tmpfile, wav_to_save, sr, encoding="PCM_S16")
        display(Audio(tmpfile, rate=sr))
        return
    except Exception as e_ta:
        last_exc = e_ta

    # Try scipy.io.wavfile.write
    try:
        from scipy.io import wavfile
        tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        tmpfile = tmp.name
        tmp.close()
        # convert to int16
        wav_int16 = (waveform * 32767.0).astype(np.int16)
        wavfile.write(tmpfile, sr, wav_int16)
        display(Audio(tmpfile, rate=sr))
        return
    except Exception as e_sp:
        last_exc = e_sp

    # If all fallbacks failed, raise with full debug info
    errmsg = (
        "All playback methods failed.\n\n"
        f"Direct playback error: {err}\n\n"
        f"Last fallback exception: {last_exc}\n\n"
        "Traceback (direct):\n" + tb
    )
    raise RuntimeError(errmsg)


Exception ignored in: <function Wave_write.__del__ at 0x725a91d59800>
Traceback (most recent call last):
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/wave.py", line 447, in __del__
    self.close()
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/wave.py", line 565, in close
    self._ensure_header_written(0)
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/wave.py", line 588, in _ensure_header_written
    self._write_header(datasize)
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/wave.py", line 592, in _write_header
    self._file.write(b'RIFF')
ValueError: I/O operation on closed file.
Exception ignored in: <function Wave_write.__del__ at 0x725a91d59800>
Traceback (most recent call last):
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/wave.py", line 447, in __del__
    self.close()
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/wave.py", line 565, in close
    self._ensure_header_written(0)
  File "/home/nampv1/anaconda3/envs/asr/lib/pyth

In [36]:
listen_audio(clean)

In [37]:
listen_audio(noise)

In [44]:
noisy_audio = torch.tensor(noisy_audio)
noisy_audio

/tmp/ipykernel_692741/3602046615.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  noisy_audio = torch.tensor(noisy_audio)


tensor([ 0.0152,  0.0137,  0.0111,  ..., -0.0020, -0.0018, -0.0018],
       dtype=torch.float64)

In [38]:
from df.enhance import enhance, init_df, load_audio, save_audio

In [115]:
audio, meta = load_audio("/home/nampv1/projects/asr/asr_ft/data/examples/audio.wav",)
audio, meta

/home/nampv1/anaconda3/envs/asr/lib/python3.11/site-packages/df/io.py:36: UserWarning: torchaudio._backend.utils.info has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  info: AudioMetaData = ta.info(file, **ikwargs)
/home/nampv1/anaconda3/envs/asr/lib/python3.11/site-packages/torchaudio/_backend/soundfile_backend.py:120: UserWarning: torchaudio._backend.common.AudioMetaData has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more inf

(tensor([[ 0.0152,  0.0137,  0.0111,  ..., -0.0020, -0.0018, -0.0018]]),
 <torchaudio._backend.common.AudioMetaData at 0x7538141b5f50>)

In [117]:
# Tạo Gaussian noise
noise_level = 0.02  # chỉnh tăng/giảm âm lượng noise
noise = torch.randn_like(audio[0]) * noise_level

# Thêm noise vào audio
noisy_audio = audio[0] + noise

# Bắt buộc shape phải [channels, samples]
if noisy_audio.ndim == 1:
    noisy_audio = noisy_audio.unsqueeze(0)  # [1, N]

# Ép dtype float32
# noisy_audio = noisy_audio.to(torch.float32)
# noisy_audio = torch.clamp(noisy_audio, -1.0, 1.0)

In [118]:
save_audio("noisy.wav", noisy_audio, sr=meta.sample_rate)

In [39]:
# from df import enhance, init_df
import torch


model, df_state, _ = init_df(
    model_base_dir = "/home/nampv1/projects/asr/asr_ft/enhance_audio/DeepFilterNet2_model/DeepFilterNet2"
)  # loads default model
noisy_audio = clean # load your wav file (e.g. via torchaudio or soundfile)


enhanced_audio = enhance(model, df_state, noisy_audio)
# Then save enhanced_audio


2025-10-01 16:50:54 | INFO     | DF | Loading model settings of DeepFilterNet2
2025-10-01 16:50:54 | INFO     | DF | Initializing model `deepfilternet2`


Exception ignored in: <function Wave_write.__del__ at 0x725a91d59800>
Traceback (most recent call last):
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/wave.py", line 447, in __del__
    self.close()
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/wave.py", line 565, in close
    self._ensure_header_written(0)
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/wave.py", line 588, in _ensure_header_written
    self._write_header(datasize)
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/wave.py", line 592, in _write_header
    self._file.write(b'RIFF')
ValueError: I/O operation on closed file.
Exception ignored in: <function Wave_write.__del__ at 0x725a91d59800>
Traceback (most recent call last):
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/wave.py", line 447, in __del__
    self.close()
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/wave.py", line 565, in close
    self._ensure_header_written(0)
  File "/home/nampv1/anaconda3/envs/asr/lib/pyth

2025-10-01 16:50:54 | INFO     | DF | Found checkpoint /home/nampv1/projects/asr/asr_ft/enhance_audio/DeepFilterNet2_model/DeepFilterNet2/checkpoints/model_96.ckpt.best with epoch 96
2025-10-01 16:50:54 | INFO     | DF | Running on device cuda:0
2025-10-01 16:50:54 | INFO     | DF | Model loaded


In [40]:
save_audio("enhanced.wav", enhanced_audio, sr)

/home/nampv1/anaconda3/envs/asr/lib/python3.11/site-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(


In [33]:
listen_audio(enhanced_audio)
